# MongoDB vs SQL

## Overview
This notebook provides a direct comparison between MongoDB (PyMongo) and PostgreSQL (SQL) -- translating common operations, explaining when each is the right choice, and introducing PostgreSQL's JSONB type as a middle path.

**The core difference:**

| | MongoDB | PostgreSQL (SQL) |
|---|---|---|
| Data model | Flexible documents (BSON/JSON) | Fixed-schema tables with rows |
| Schema | Enforced by application (optional `$jsonSchema`) | Enforced by DDL (`CREATE TABLE`) |
| Relationships | Embedded documents or references + `$lookup` | Foreign keys + `JOIN` |
| Transactions | Single-document atomic; multi-document ACID since 4.0 | Full ACID across any tables |
| Scaling | Horizontal sharding built-in | Vertical primary; logical replication; Citus for sharding |
| Query language | MongoDB Query Language (MQL) + aggregation pipeline | SQL (ANSI standard) |
| Analytics tooling | Limited native; Atlas Charts or export to SQL | Excellent; every BI tool speaks SQL |

**Both are mature, production-grade databases.** The choice depends on data structure, access patterns, team expertise, and operational constraints.

---

In [ ]:

print("=== Side-by-side: SQL vs MongoDB ===")
comparisons = [
    ("Create table/collection",
     "CREATE TABLE patients (\n  patient_id VARCHAR PRIMARY KEY,\n  first_name VARCHAR NOT NULL,\n  sex CHAR(1)\n);",
     "db.create_collection('patients')\n# or just: patients = db['patients']\n# Schema is implicit; no DDL required"),
    ("Insert",
     "INSERT INTO patients VALUES ('P001','Aroha','F','NB');",
     "patients.insert_one({'patient_id':'P001','first_name':'Aroha','sex':'F','province':'NB'})"),
    ("Select all",
     "SELECT * FROM patients;",
     "patients.find({})"),
    ("Filter",
     "SELECT * FROM patients WHERE province = 'NB';",
     "patients.find({'province': 'NB'})"),
    ("Count",
     "SELECT COUNT(*) FROM patients WHERE province = 'NB';",
     "patients.count_documents({'province': 'NB'})"),
    ("Projection",
     "SELECT first_name, last_name FROM patients WHERE province = 'NB';",
     "patients.find({'province':'NB'},{'first_name':1,'last_name':1,'_id':0})"),
    ("Group by",
     "SELECT province, COUNT(*) FROM patients GROUP BY province;",
     "patients.aggregate([{'$group':{'_id':'$province','count':{'$sum':1}}}])"),
    ("Join",
     "SELECT p.first_name, e.encounter_date\nFROM patients p JOIN encounters e\nON p.patient_id = e.patient_id;",
     "encounters.aggregate([\n  {'$lookup':{'from':'patients','localField':'patient_id',\n   'foreignField':'patient_id','as':'patient'}},\n  {'$unwind':'$patient'},\n  {'$project':{'encounter_date':1,'first_name':'$patient.first_name'}}\n])"),
    ("Update",
     "UPDATE patients SET province = 'NS' WHERE patient_id = 'P001';",
     "patients.update_one({'patient_id':'P001'},{'$set':{'province':'NS'}})"),
    ("Delete",
     "DELETE FROM patients WHERE patient_id = 'P001';",
     "patients.delete_one({'patient_id':'P001'})"),
    ("Create index",
     "CREATE INDEX ON patients (province);",
     "patients.create_index('province')"),
]
for label, sql, mongo in comparisons:
    print(f"  --- {label} ---")
    print(f"  SQL:    {sql}")
    print(f"  Mongo:  {mongo}")
    print()


---
## When to choose MongoDB vs PostgreSQL

In [ ]:

print("=== When to choose MongoDB vs PostgreSQL ===")
rows = [
    ("Use MongoDB",   "Variable document structure per record (polymorphic data)"),
    ("Use MongoDB",   "Hierarchical nested data that maps naturally to JSON"),
    ("Use MongoDB",   "High write throughput; horizontal sharding across many servers"),
    ("Use MongoDB",   "Geospatial queries with GeoJSON (though PostGIS is more mature)"),
    ("Use MongoDB",   "Full-text search on large text fields (though Elasticsearch is better for this)"),
    ("Use MongoDB",   "Rapid prototyping where schema requirements are still changing"),
    ("Use MongoDB",   "Event/log storage where each event type has different fields"),
    ("Use PostgreSQL","Multi-table ACID transactions (ORDER + PAYMENT + INVENTORY atomically)"),
    ("Use PostgreSQL","Complex analytical queries across many related tables"),
    ("Use PostgreSQL","Strong referential integrity requirements (FK constraints)"),
    ("Use PostgreSQL","Regulatory environments requiring strict schema enforcement and audit logs"),
    ("Use PostgreSQL","Existing analyst/BI tooling that expects SQL and tabular output"),
    ("Use PostgreSQL","JSONB columns give 80% of MongoDB flexibility within a SQL database"),
    ("Use either",    "Time-series data (both have dedicated optimizations now)"),
    ("Use either",    "Content management systems"),
    ("Use either",    "REST API backends (both have excellent driver support)"),
]
print(f"  {'Decision':<18s}  Use case")
print("  " + "-"*72)
for decision, use_case in rows:
    print(f"  {decision:<18s}  {use_case}")


---
## PostgreSQL JSONB: a middle path

In [ ]:

print("=== PostgreSQL JSONB: SQL with MongoDB-style flexibility ===")
jsonb_code = '''
-- PostgreSQL JSONB lets you store and query JSON inside a SQL column.
-- This gives you much of MongoDB's document flexibility with full SQL support.

-- Store flexible patient data as JSONB:
CREATE TABLE patients (
    patient_id   VARCHAR PRIMARY KEY,
    first_name   VARCHAR NOT NULL,
    last_name    VARCHAR NOT NULL,
    demographics JSONB,        -- flexible fields stored as JSON
    conditions   JSONB         -- array of conditions as JSON
);

INSERT INTO patients VALUES (
    'P001', 'Aroha', 'Ngata',
    '{"dob": "1985-03-15", "sex": "F", "province": "NB"}',
    '["hypertension", "type2_diabetes"]'
);

-- Query JSONB fields:
SELECT first_name, demographics->>'province' AS province
FROM patients
WHERE demographics->>'province' = 'NB';

-- Query JSON array (conditions contains 'hypertension'):
SELECT * FROM patients
WHERE conditions @> '["hypertension"]';

-- Index a JSONB field:
CREATE INDEX ON patients ((demographics->>'province'));

-- GIN index for containment queries:
CREATE INDEX ON patients USING GIN (conditions);
-- Now: WHERE conditions @> '["hypertension"]' uses the index
'''
print(jsonb_code)

print("When JSONB is the right choice:")
jsonb_use_cases = [
    "You need mostly relational data but a few flexible/variable columns",
    "You want SQL JOINs, transactions, and analytics alongside some JSON flexibility",
    "Your team knows SQL well and does not want to learn a new query language",
    "You are already on PostgreSQL -- no new infrastructure required",
]
for u in jsonb_use_cases:
    print(f"  - {u}")


---
## Common Pitfalls

**1. Choosing MongoDB to avoid schema design work**
MongoDB's flexible schema does not eliminate the need for schema design -- it just defers the consequences. A collection where every document has a different shape becomes impossible to query consistently and expensive to index. Schema design is required in MongoDB; it just happens in application code and indexes rather than DDL.

**2. Using MongoDB for everything because it is 'modern'**
MongoDB is the right tool for specific workloads. For a finance or healthcare system with complex multi-entity transactions and strict integrity requirements, PostgreSQL is almost always the better choice. Using MongoDB for a relational workload leads to re-implementing JOIN, transaction, and constraint logic in application code -- which is harder than just using SQL.

**3. Treating MongoDB's lack of JOINs as a limitation to work around**
If you find yourself writing many `$lookup` stages to join 5 collections, your data is relational and belongs in a relational database. MongoDB is designed for data that is naturally hierarchical and accessed as a unit. Frequent cross-collection joins signal a schema that should be in SQL.

**4. Not understanding ACID transaction scope**
MongoDB supports multi-document ACID transactions (since 4.0) but they are more expensive than single-document operations. The document model is specifically designed to allow most operations to be completed in a single document write (which is atomic). Requiring many multi-document transactions defeats the purpose of the document model.

**5. Ignoring PostgreSQL JSONB as a middle path**
PostgreSQL's JSONB column type stores and indexes JSON documents, supports containment and existence operators, and can be queried with GIN indexes almost as fast as MongoDB. For most teams on PostgreSQL who want some document flexibility, JSONB is simpler than adding a separate MongoDB deployment.

**6. Assuming horizontal scaling is automatic**
MongoDB sharding (horizontal scaling) requires careful shard key selection. A bad shard key creates hot spots where one shard receives all the traffic. Sharding also makes multi-shard transactions and `$lookup` across shards very expensive. Plan the shard key based on access patterns before the collection exceeds a single server's capacity.


---
*sql_methods_library - Samantha McGarrigle*